In [3]:
from dotenv import load_dotenv
import os
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

from sklearn.metrics import accuracy_score

In [4]:
load_dotenv()

print("Key geladen?", "GEMINI_API_KEY" in os.environ)

Key geladen? True


In [5]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.0-flash"

/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: 
    You are using a Python version 3.9 past its end of life. Google will update
    google-auth with critical bug fixes on a best-effort basis, but not
    with any other fixes or features. Please upgrade your Python version,
    and then update google-auth.
    
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lennartredlich/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: 
    You are using a Python version 3.9 past its end of life. Google will update
    google-auth with critical bug fixes on a best-effort basis, but not
    with any other fixe

In [8]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs_all = []
jobs_active = []

for person_id, cv in enumerate(cvs):
    for job in cv:
        row = {**job, "person_id": person_id}
        jobs_all.append(row)
        if job.get("status") == "ACTIVE":
            jobs_active.append(row)

df_all = pd.DataFrame(jobs_all)
df_active = pd.DataFrame(jobs_active)

df_active.shape, df_all.shape

((623, 9), (2638, 9))

In [9]:
def _norm_dep(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x if x else np.nan

df_all["department_norm"] = df_all["department"].apply(_norm_dep)

In [10]:
hist = df_all[df_all["department_norm"].notna() & (df_all["department_norm"] != "Other")].copy()

In [11]:
hist_mode = (
    hist.groupby("person_id")["department_norm"]
        .agg(lambda s: s.value_counts().idxmax())
        .to_dict()
)

df_active["department_norm"] = df_active["department"].apply(_norm_dep)

In [13]:
df_active["department_gt_resolved"] = df_active["department_norm"]
needs_fix = df_active["department_gt_resolved"].isna() | (df_active["department_gt_resolved"] == "Other")
df_active.loc[needs_fix, "department_gt_resolved"] = df_active.loc[needs_fix, "person_id"].map(hist_mode)

print("\nResolved GT availability on ACTIVE:")
print(df_active["department_gt_resolved"].value_counts(dropna=False).head(20))


Resolved GT availability on ACTIVE:
department_gt_resolved
NaN                       164
Information Technology     96
Consulting                 78
Sales                      69
Project Management         58
Business Development       38
Marketing                  37
Administrative             29
Human Resources            26
Purchasing                 18
Customer Support           10
Name: count, dtype: int64


In [23]:
eval_mask = df_active["department_gt_resolved"].notna() & (df_active["department_gt_resolved"] != "Other")

df_active_fix = df_active.loc[eval_mask, [
    "position",
    "organization",
    "department_gt_resolved",
]]

df_active_fix.head(5)

,position,organization,department_gt_resolved
5,Solutions Architect,Computer Solutions,Information Technology
6,Medizintechnik Beratung,Udo Weber,Consulting
7,Director expansión de negocio.,Grupo Viajes Kontiki.,Business Development
8,Gerente comercial,Air & Ground Operations Consultancy,Sales
9,Administrador Unico,Viajes Oceano S.L.,Administrative


In [15]:
DEPARTMENT_LABELS = [
    "Information Technology",
    "Consulting",
    "Sales",
    "Project Management",
    "Business Development",
    "Marketing",
    "Administrative",
    "Human Resources",
    "Purchasing",
    "Customer Support"
]


In [24]:
SYSTEM = (
    "You are a classifier. "
    f"Return exactly ONE label from: {', '.join(DEPARTMENT_LABELS)}. "
    "Output ONLY the label."
)

In [27]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

preds = []

for _, r in tqdm(df_active_fix.iterrows(), total=len(df_active_fix)):
    prompt = f"""Job title: {r["position"]}
Company: {r["organization"]}
Which department label applies best? Choose from: {", ".join(DEPARTMENT_LABELS)}
"""
    resp = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config={"system_instruction": SYSTEM}
    )
    preds.append(resp.text.strip())

df_active_fix["department_pred"] = preds

acc = accuracy_score(df_active_fix["department_gt_resolved"], df_active_fix["department_pred"])
print("Accuracy:", round(acc, 4))

100%|██████████| 459/459 [03:40<00:00,  2.08it/s]

Accuracy: 0.6122


In [28]:
cm = pd.crosstab(
    df_active_fix["department_gt_resolved"],
    df_active_fix["department_pred"],
    normalize="index"
).round(3)

cm

department_pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Project Management,Purchasing,Sales
department_gt_resolved,,,,,,,,,,
Administrative,0.724,0.103,0.034,0.034,0.000,0.034,0.000,0.034,0.000,0.034
Business Development,0.000,0.605,0.342,0.000,0.000,0.000,0.000,0.026,0.000,0.026
Consulting,0.077,0.205,0.538,0.000,0.064,0.051,0.013,0.026,0.000,0.026
Customer Support,0.200,0.100,0.000,0.600,0.000,0.100,0.000,0.000,0.000,0.000
Human Resources,0.038,0.077,0.038,0.000,0.654,0.000,0.115,0.038,0.000,0.038
Information Technology,0.052,0.115,0.052,0.010,0.000,0.646,0.000,0.094,0.031,0.000
Marketing,0.081,0.243,0.081,0.000,0.000,0.108,0.459,0.000,0.000,0.027
Project Management,0.034,0.172,0.017,0.000,0.017,0.034,0.034,0.638,0.034,0.017
Purchasing,0.111,0.111,0.000,0.056,0.000,0.000,0.000,0.000,0.722,0.000
